# Pooled vs. slide-aware parallel-curves test (legnini)

`comparison_legnini.ipynb` pooled all cells of a condition, so the model's uncertainty shrinks with the ~40,000 cells while the real replicate count is 6 vs. 11 slides. Slide-to-slide differences then look like condition effects: splitting the SHH slides at random into two halves (no true difference) flagged 94% of genes at `tau=0` and 62% at `tau=0.1`.

Here the same binned test is run twice on the same genes:

1. **pooled**: one curve per condition, cells treated as independent (as before);
2. **slide-aware** (`bin_obs(sample="sample")`): bins per slide, plus a random level and a random curve per slide (non-centred, half-normal scales), so the condition curves are compared against the variation *between slides*. The p-value uses an F reference with `n_slides - 2` denominator degrees of freedom (column `parallel_df_denominator`). The fitted curves are the population-level ones.

Both use NUTS on bins. Variational inference is not available for the slide-aware model yet (its approximation fails the built-in diagnostics; see `.claude/advi_plan.md`).

In [ ]:
# One BLAS thread per process: the fits run in worker processes, and oversubscribed BLAS threads slow them down a lot.
import os

for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[var] = "1"

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq

import diffsvg as dvg

In [ ]:
adata = sc.read_h5ad("/home/lehnerl/Arbeit/data/legnini23.h5ad")

shh = adata[:, "SHH"].X
shh = (shh.toarray() if hasattr(shh, "toarray") else np.asarray(shh)).ravel()
adata.obs["SHH+"] = (shh > np.percentile(shh, 99)).astype(str)

sq.tl.var_by_distance(adata, cluster_key="SHH+", groups="True", library_key="sample", covariates="condition")
adata.obsm["design_matrix"].head()

## Settings

The slide-aware model has ~10 extra parameters per slide, so NUTS needs a little more room than for the pooled model: check `rhat_max` (ideally below 1.05), `ess_bulk_min` and `n_divergent` in the tables below. Near-empty genes (almost only zeros) carry no information about slide variation and may not converge; their p-values should not be read.

In [ ]:
LAYER = "log1p_norm"
GENES = list(adata.var_names)  # shorten for a trial run
N_WORKERS = 4
TAU = 0.1
SAMPLER = {"chains": 4, "draws": 1000, "tune": 1000}
PARALLEL_COLUMNS = [
    "shape_diff", "slope_diff", "parallel_stat", "parallel_pval", "parallel_pval_adj", "integral",
    "n_divergent", "rhat_max", "ess_bulk_min",
]  # fmt: skip


def rank_table(stats, columns):
    """Genes ordered by p-value, ties (e.g. p-values that underflow to 0) broken by the larger statistic."""
    table = stats.sort_values(["parallel_pval", "parallel_stat"], ascending=[True, False])[columns].copy()
    table.insert(0, "rank", np.arange(1, len(table) + 1))
    return table


def qc_plots(stats, title):
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    dvg.pl.plot_pvalue_histogram(stats, ax=axes[0])
    dvg.pl.plot_volcano(stats, ax=axes[1], n_label=15)
    fig.suptitle(title)
    fig.tight_layout()
    return fig

## 1. Pooled (as in `comparison_legnini.ipynb`)

In [ ]:
t0 = time.perf_counter()
dvg.pp.bin_obs(adata, anchor="True", condition="condition", genes=GENES, layer=LAYER, n_bins=50, window_radius=None)
pooled_stats, _, _ = dvg.tl.fit_model(
    adata, dvg.tl.setup_model(adata), n_workers=N_WORKERS, test="parallel", tau=TAU, inplace=False, **SAMPLER
)
print(f"{time.perf_counter() - t0:.0f} s")
pooled_table = rank_table(pooled_stats, PARALLEL_COLUMNS)
pooled_table.head(25)

In [ ]:
qc_plots(pooled_stats, f"Pooled, parallel-curves test (tau = {TAU})");

## 2. Slide-aware

`sample="sample"` names the slide column of the design matrix; every slide has to belong to one condition. Slides without any anchor cell (`slide1_A2-2`) have no distances and drop out of the fit.

In [ ]:
t0 = time.perf_counter()
dvg.pp.bin_obs(
    adata, anchor="True", condition="condition", genes=GENES, layer=LAYER, n_bins=50, window_radius=None, sample="sample"
)
aware_stats, _, aware_curves = dvg.tl.fit_model(
    adata, dvg.tl.setup_model(adata), n_workers=N_WORKERS, test="parallel", tau=TAU, inplace=False, **SAMPLER
)
print(f"{time.perf_counter() - t0:.0f} s")
aware_table = rank_table(aware_stats, [*PARALLEL_COLUMNS, "parallel_df_denominator"])
aware_table.head(25)

In [ ]:
qc_plots(aware_stats, f"Slide-aware, parallel-curves test (tau = {TAU})");

In [ ]:
poor = aware_stats[(aware_stats["rhat_max"] > 1.05) | (aware_stats["n_divergent"] > 0)]
print(f"{len(poor)} of {len(aware_stats)} genes did not sample cleanly (rhat_max > 1.05 or divergences); read their p-values with care")
poor[["rhat_max", "ess_bulk_min", "n_divergent", "parallel_pval"]].sort_values("rhat_max", ascending=False).head(15)

## Comparison

In [ ]:
summary = pd.DataFrame(
    {
        "significant at 0.05": {
            "pooled": int((pooled_stats["parallel_pval"] < 0.05).sum()),
            "slide-aware": int((aware_stats["parallel_pval"] < 0.05).sum()),
        },
        "significant after BH": {
            "pooled": int((pooled_stats["parallel_pval_adj"] < 0.05).sum()),
            "slide-aware": int((aware_stats["parallel_pval_adj"] < 0.05).sum()),
        },
    }
)
display(summary)

ranks = pd.DataFrame({"pooled": pooled_table["rank"], "slide-aware": aware_table["rank"]})
print("Spearman correlation of the ranks:", round(ranks.corr(method="spearman").iloc[0, 1], 3))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].scatter(pooled_stats["shape_diff"], aware_stats.loc[pooled_stats.index, "shape_diff"], s=12)
lim = float(max(pooled_stats["shape_diff"].max(), aware_stats["shape_diff"].max())) * 1.05
axes[0].plot([0, lim], [0, lim], color="grey", lw=0.8)
axes[0].set(xlabel="shape_diff, pooled", ylabel="shape_diff, slide-aware", title="Effect sizes")
floor = 1e-300
axes[1].scatter(
    -np.log10(pooled_stats["parallel_pval"].clip(lower=floor)),
    -np.log10(aware_stats.loc[pooled_stats.index, "parallel_pval"].clip(lower=floor)),
    s=12,
)
axes[1].axhline(-np.log10(0.05), color="grey", lw=0.8)
axes[1].set(xlabel="-log10 p, pooled", ylabel="-log10 p, slide-aware", title="Significance")
fig.tight_layout()

## Optional: how often does each version call a difference where there is none?

Splits the SHH slides at random into two halves (same condition, so every rejection is false) and counts rejections at `alpha = 0.05`. This refits the panel for every split, so it is off by default and restricted to 10 genes and 3 splits; the earlier check (10 genes, 5 splits) gave 94% / 62% pooled (`tau = 0` / `0.1`) and 4% / 0% slide-aware.

In [ ]:
RUN_SPLIT_NULL = False  # set to True to run

if RUN_SPLIT_NULL:
    rng = np.random.default_rng(0)
    shh_obs = (adata.obs["condition"].astype(str) == "SHH").to_numpy() & adata.obsm["design_matrix"]["True"].notna().to_numpy()
    null_genes = GENES[:10]
    sub = adata[shh_obs, null_genes].copy()
    slides = np.array(sorted(sub.obs["sample"].astype(str).unique()))
    rows = []
    for split in range(3):
        half = set(rng.choice(slides, len(slides) // 2, replace=False))
        design = sub.obsm["design_matrix"].copy()
        design["condition"] = sub.obs["sample"].astype(str).map(lambda s: "A" if s in half else "B").to_numpy()
        sub.obsm["design_matrix"] = design
        for label, sample in [("pooled", None), ("slide-aware", "sample")]:
            dvg.pp.bin_obs(
                sub, anchor="True", condition="condition", genes=null_genes, layer=LAYER, n_bins=50, window_radius=None, sample=sample
            )
            for tau in (0.0, TAU):
                stats, _, _ = dvg.tl.fit_model(
                    sub, dvg.tl.setup_model(sub), n_workers=N_WORKERS, test="parallel", tau=tau, inplace=False, **SAMPLER
                )
                rows.append({"model": label, "tau": tau, "split": split, "rejected": float((stats["parallel_pval"] < 0.05).mean())})
    display(pd.DataFrame(rows).groupby(["model", "tau"])["rejected"].mean().unstack())